# AI-Powered Q&A System with Memory
### Phase 2 Mini-Project · LangChain + Google Gemini + Streamlit
---
| Detail | Value |
|---|---|
| **Framework** | LangChain 0.3+ (LCEL) |
| **LLM** | Google Gemini 2.5 Flash via AI Studio |
| **Memory** | `RunnableWithMessageHistory` (modern 0.3+ API) |
| **UI** | Streamlit (standalone `qa_app.py`) |



## Step 1 — Install Dependencies

In [ ]:
# ─── Install compatible package set ──────────────────────────────────────────
# NOTE: langchain-google-genai ≥ 2.0 requires langchain ≥ 0.3.
#       The old langchain.memory / langchain.chains paths are GONE in 0.3.
#       We use RunnableWithMessageHistory + ChatMessageHistory instead.

import subprocess, sys

PACKAGES = [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-google-genai",
    "google-generativeai",
    "python-dotenv",
    "streamlit>=1.40.0",
    "pydantic>=2.0",
]

print("Installing packages…")
for pkg in PACKAGES:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--upgrade"],
        capture_output=True, text=True
    )
    name = pkg.split(">=")[0]
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status} {name}")

print("\nAll packages installed! → Restart runtime now (Runtime > Restart session) then continue.")

## Step 2 — API Key Setup

Get your **free** key from [Google AI Studio](https://aistudio.google.com/prompts/new_chat):


In [ ]:
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Google AI Studio API key: ")

key = os.environ["GOOGLE_API_KEY"]
if len(key) > 10:
    print(f"Key loaded — {key[:6]}{'*' * (len(key) - 10)}{key[-4:]}")
else:
    print(" Key looks short — double-check it.")

## Step 3 — Imports & Configuration

**Why these imports?**
- `RunnableWithMessageHistory` — LangChain 0.3+ replacement for `ConversationChain`
- `ChatMessageHistory` — in-memory store for conversation turns (replaces `ConversationBufferMemory`)
- `InMemoryChatMessageHistory` — lightweight alternative from `langchain_core`
- `ChatPromptTemplate` + `MessagesPlaceholder` — injects history into every prompt

In [ ]:
import os, time, textwrap
from datetime import datetime
from typing import Dict

# LLM
from langchain_google_genai import ChatGoogleGenerativeAI

# Prompt building
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Output parsing
from langchain_core.output_parsers import StrOutputParser

# Message types
from langchain_core.messages import HumanMessage, AIMessage

# Modern memory (LangChain 0.3+) — replaces ConversationBufferMemory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

# ─── Global config ─────────────────────────────────────────────────────────────
MODEL_NAME   = "gemini-2.5-flash"
DEFAULT_TEMP = 0.7
MAX_TOKENS   = 2048

SYSTEM_PROMPT = """You are an expert AI assistant.
Rules:
1. ACCURACY  — Only state confident facts. Say "I'm not sure" otherwise.
2. CLARITY   — Use bullet points or code blocks where helpful.
3. CONTEXT   — Reference prior messages when relevant.
4. BREVITY   — Complete but concise. No filler phrases.
5. TONE      — Professional yet approachable.

Session date: {today}"""

print("Imports loaded")
print(f"   Model : {MODEL_NAME}")
print(f"   Temp  : {DEFAULT_TEMP}")
print(f"   Using : RunnableWithMessageHistory (LangChain 0.3+ memory API)")

## Step 4 — Build the LLM Chain

LCEL pipeline: `prompt | llm | parser`

Wrapped with `RunnableWithMessageHistory` so every call automatically:
1. Loads history from the store
2. Injects it into the prompt
3. Appends the new turn after the response

In [ ]:
# ─── 1. LLM
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=DEFAULT_TEMP,
    max_output_tokens=MAX_TOKENS,
)

# ─── 2. Prompt template ───────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT.format(today=datetime.now().strftime("%B %d, %Y"))),
    MessagesPlaceholder(variable_name="history"),   # ← conversation history injected here
    ("human", "{input}"),
])

# ─── 3. Chain (LCEL) ──────────────────────────────────────────────────────────
chain = prompt | llm | StrOutputParser()

# ─── 4. In-memory session store ───────────────────────────────────────────────
# Maps session_id → ChatMessageHistory object
_store: Dict[str, BaseChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in _store:
        _store[session_id] = ChatMessageHistory()
    return _store[session_id]

# ─── 5. Wrap chain with automatic history management ──────────────────────────
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

print("Chain built")
print("   pipeline : ChatPromptTemplate | ChatGoogleGenerativeAI | StrOutputParser")
print("   memory   : RunnableWithMessageHistory (per-session store)")

##  Step 5 — QASystem Class

In [ ]:
class QASystem:
    """Thin wrapper around chain_with_history for clean multi-session usage."""

    def __init__(self, session_id: str = "default"):
        self.session_id = session_id
        self.turn_count = 0

    def ask(self, question: str, verbose: bool = True) -> str:
        t0 = time.time()
        response = chain_with_history.invoke(
            {"input": question},
            config={"configurable": {"session_id": self.session_id}},
        )
        elapsed = round(time.time() - t0, 2)
        self.turn_count += 1

        if verbose:
            print(f"\n{'─'*60}")
            print(f"[Turn {self.turn_count}] Q: {question}")
            print(f"{'─'*60}")
            for line in textwrap.wrap(response, 70):
                print(" ", line)
            print(f"  ⏱  {elapsed}s")

        return response

    def get_history(self) -> list:
        return get_session_history(self.session_id).messages

    def clear(self):
        _store.pop(self.session_id, None)
        self.turn_count = 0
        print(f"Session '{self.session_id}' cleared.")

    def export(self, filename: str = None) -> str:
        """Export conversation to a .txt file."""
        filename = filename or f"qa_export_{self.session_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        lines = []
        for msg in self.get_history():
            role = "USER" if isinstance(msg, HumanMessage) else "AI"
            lines.append(f"[{role}] {msg.content}\n")
        with open(filename, "w") as f:
            f.writelines(lines)
        print(f"Exported {len(lines)} messages → {filename}")
        return filename

print("QASystem class ready")

## Step 6 — Memory Test (4-turn conversation)

In [ ]:
qa = QASystem(session_id="memory_test")

qa.ask("Hi! My name is Ali and I'm learning LangChain.")
qa.ask("What is LangChain and why is it useful?")
qa.ask("What are the core components of LangChain?")
qa.ask("What's my name and what am I learning?")

print(f"\n 4-turn memory test complete. Total turns: {qa.turn_count}")
print(f"   Messages in history: {len(qa.get_history())}  (8 = 4 human + 4 AI)")

##  Step 7 — Batch Q&A + Few-Shot Prompting Demo

In [ ]:
# ─── Batch Q&A ────────────────────────────────────────────────────────────────
batch_qa = QASystem(session_id="batch")

questions = [
    "What is RAG (Retrieval-Augmented Generation)?",
    "How does RAG differ from fine-tuning?",
    "Give me a one-line Python example showing how a LangChain chain is built.",
]

print("=" * 60)
print("BATCH Q&A MODE")
print("=" * 60)

for q in questions:
    batch_qa.ask(q)

print(f"\nBatch complete — {batch_qa.turn_count} questions answered")

In [ ]:
# ─── Few-Shot Prompting Demo ──────────────────────────────────────────────────
# Shows how to bake examples directly into a prompt template.

# Re-initializing llm locally for this cell to ensure the correct model is used,
# as the global 'llm' object might not have been updated after changing MODEL_NAME
# in the 'imports-cell' if 'chain-cell' was not re-executed.
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=DEFAULT_TEMP,
    max_output_tokens=MAX_TOKENS,
)

few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical explainer. Always answer in ≤ 3 sentences."),
    # Few-shot examples baked into the prompt
    ("human", "What is an API?"),
    ("ai",    "An API (Application Programming Interface) is a contract that lets two software systems talk to each other. It defines the requests you can make, how to make them, and the format of responses. REST and GraphQL are common API styles."),
    ("human", "What is a vector database?"),
    ("ai",    "A vector database stores data as high-dimensional numerical embeddings. It excels at similarity search — finding items that are semantically close to a query. Popular options include Chroma, Pinecone, and FAISS."),
    # Actual user question
    ("human", "{input}"),
])

few_shot_chain = few_shot_prompt | llm | StrOutputParser()

answer = few_shot_chain.invoke({"input": "What is LangChain?"})
print("\nFew-Shot Prompt Result:")
print("-" * 50)
print(answer)

##  Step 8 — Streamlit App

Writes `qa_app.py` then runs it.  
- **Local**: `streamlit run qa_app.py`  
- **Colab**: run the launcher cell below it

In [ ]:
code = """
import os, time
from datetime import datetime
from typing import Dict

import streamlit as st
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

MODEL = "gemini-2.5-flash"
MAX_TOKENS = 2048

st.set_page_config(page_title="AI Q&A System", page_icon="🤖", layout="wide")

if "messages" not in st.session_state:
    st.session_state.messages = []
if "history_store" not in st.session_state:
    st.session_state.history_store = {}
if "turn_count" not in st.session_state:
    st.session_state.turn_count = 0
if "connected" not in st.session_state:
    st.session_state.connected = False

def get_session_history(session_id):
    store = st.session_state.history_store
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

def build_chain(api_key, temp):
    llm = ChatGoogleGenerativeAI(model=MODEL, temperature=temp,
                                  max_output_tokens=MAX_TOKENS, google_api_key=api_key)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI assistant. Today: " + datetime.now().strftime("%B %d, %Y")),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ])
    return RunnableWithMessageHistory(
        prompt | llm | StrOutputParser(),
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

with st.sidebar:
    api_key = st.text_input("API Key", type="password", placeholder="AIza...")
    if api_key and not st.session_state.connected:
        with st.spinner("Verifying..."):
            try:
                ChatGoogleGenerativeAI(model=MODEL, google_api_key=api_key).invoke([HumanMessage(content="hi")])
                st.session_state.connected = True
            except Exception as e:
                st.error(str(e)[:100])
    if api_key and st.session_state.connected:
        st.success("Connected")
    temperature = st.slider("Temperature", 0.0, 1.0, 0.7, 0.05)
    st.write("Turns:", st.session_state.turn_count)
    if st.button("Clear"):
        st.session_state.messages = []
        st.session_state.history_store = {}
        st.session_state.turn_count = 0
        st.rerun()

st.title("AI Q&A System")

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

if question := st.chat_input("Ask anything..."):
    if not api_key:
        st.error("Enter API key in sidebar.")
        st.stop()
    if not st.session_state.connected:
        st.error("API key not valid.")
        st.stop()

    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.write(question)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                answer = build_chain(api_key, temperature).invoke(
                    {"input": question},
                    config={"configurable": {"session_id": "default"}}
                )
                st.write(answer)
                st.session_state.messages.append({"role": "assistant", "content": answer})
                st.session_state.turn_count += 1
            except Exception as e:
                st.error(str(e))
"""

with open("qa_app.py", "w") as f:
    f.write(code)

print("qa_app.py written successfully!")

In [ ]:
!pip install pyngrok


In [ ]:
from pyngrok import ngrok

In [ ]:
import subprocess
import threading
import time

def run():
    subprocess.Popen([
        "streamlit",
        "run",
        "qa_app.py",
        "--server.port=8501"
    ])

threading.Thread(target=run).start()

time.sleep(5)

In [ ]:
public_url = ngrok.connect(8501)

print(public_url)

## Step 10 — Completion Checklist

In [ ]:
checklist = [
    ("Dependencies installed (no version conflicts)",         True),
    ("Google Gemini API connected",                           True),
    ("ChatPromptTemplate with system prompt",                 True),
    ("LCEL chain  (prompt | llm | parser)",                   True),
    ("RunnableWithMessageHistory  (modern memory API)",       True),
    ("QASystem class (OOP wrapper)",                          True),
    ("4-turn memory test — model remembers context",          True),
    ("Few-shot prompting demo",                               True),
    ("Batch Q&A mode",                                        True),
    ("Streamlit app written (qa_app.py)",                     True),
    ("Colab public URL launcher",                             True),
    ("Export conversation history",                           True),
]

print("\nPHASE 2 PROJECT — COMPLETION REPORT")
print("=" * 52)
passed = sum(1 for _, v in checklist if v)
for label, done in checklist:
    print(f"  {'✅' if done else '❌'}  {label}")
print("=" * 52)
print(f"  {passed}/{len(checklist)} complete")
print()
print("Phase 2 DONE. Ready for Phase 3 (Agents & RAG)!")